# Pseudotime Trajectory Inference

## Imports

In [1]:
import anndata as ad
import anndata2ri
import numpy as np
import pandas as pd
import scanpy as sc
from rpy2.robjects.packages import importr

%load_ext rpy2.ipython
anndata2ri.set_ipython_converter()

In [2]:
%%R
library(SingleCellExperiment)
library(slingshot)
library(RaceID)

cat("R OK\n")
sessionInfo()

R OK
R version 4.4.3 (2025-02-28)
Platform: x86_64-conda-linux-gnu
Running under: Pop!_OS 22.04 LTS

Matrix products: default
BLAS/LAPACK: /home/jasiek/anaconda3/envs/scrna-ti/lib/libopenblasp-r0.3.34.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=pl_PL.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=pl_PL.UTF-8        LC_COLLATE=pl_PL.UTF-8    
 [5] LC_MONETARY=pl_PL.UTF-8    LC_MESSAGES=pl_PL.UTF-8   
 [7] LC_PAPER=pl_PL.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=pl_PL.UTF-8 LC_IDENTIFICATION=C       

time zone: Europe/Warsaw
tzcode source: system (glibc)

attached base packages:
[1] stats4    tools     stats     graphics  grDevices utils     datasets 
[8] methods   base     

other attached packages:
 [1] RaceID_0.4.2                slingshot_2.14.0           
 [3] TrajectoryUtils_1.14.0      princurve_2.1.6            
 [5] SingleCellExperiment_1.28.0 SummarizedExperiment_1.36.0
 [7] Biobase_2.66.0  

Ładowanie wymaganego pakietu: SummarizedExperiment
Ładowanie wymaganego pakietu: MatrixGenerics
Ładowanie wymaganego pakietu: matrixStats

Dołączanie pakietu: ‘MatrixGenerics’

Następujące obiekty zostały zakryte z ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, rowCounts, rowCummaxs, rowCummins, rowCumprods,
    rowCumsums, rowDiffs, rowIQRDiffs, rowIQRs, rowLogSumExps,
    rowMadDiffs, rowMads, rowMaxs, rowMeans2, rowMedians, rowMins,
    rowOrderStats, rowProds, rowQuantiles, 

### Data Loading

In [3]:
adata = sc.read("merged_dataset.h5ad")
adata

AnnData object with n_obs × n_vars = 14915 × 28692
    obs: 'batch', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt'
    var: 'mt'

## Common preprocessing

In [ ]:
# Metadata from Cell Annotation (+ optional Hérault paper columns for Layer B)
meta = pd.read_csv("cell_annotation_metadata.tsv", sep="\t", index_col=0)
assert meta.index.equals(adata.obs_names) or meta.index.isin(adata.obs_names).all()
adata.obs = adata.obs.join(meta[["label", "label_color", "leiden", "discard"]], how="left")

batch_to_sample = {
    "young_A": "young_A",
    "young_B": "young_B",
    "old_A": "aged_A",
    "old_B": "aged_B",
}
_barcode = pd.Index(adata.obs_names).str.replace(r"_\d+$", "", regex=True)
_sample = adata.obs["batch"].astype(str).map(batch_to_sample)
_paper_key = pd.Index(_sample.to_numpy() + "_" + _barcode.to_numpy())

paper = pd.read_csv(
    "sc_metadata.csv",
    usecols=["barcode", "sampleName", "HSPC_state", "Pseudotime", "hscScore", "Monocle_state"],
)
paper = paper.assign(paper_key=paper["sampleName"].astype(str) + "_" + paper["barcode"].astype(str))
paper = paper.drop_duplicates("paper_key").set_index("paper_key")
paper_aligned = paper.reindex(_paper_key)
paper_aligned.index = adata.obs_names
adata.obs["paper_state"] = paper_aligned["HSPC_state"].values
adata.obs["paper_pseudotime"] = paper_aligned["Pseudotime"].values
adata.obs["hscScore"] = paper_aligned["hscScore"].values
adata.obs["monocle_state"] = paper_aligned["Monocle_state"].values

n_before = adata.n_obs
adata = adata[~adata.obs["discard"].fillna(False).astype(bool)].copy()
print(f"Dropped discard=True: {n_before - adata.n_obs} cells → {adata.n_obs} left")
print("labels:", adata.obs["label"].value_counts().to_dict())
print("paper_state:", adata.obs["paper_state"].value_counts(dropna=False).head(12).to_dict())

### Layer A — biological priors from labels

Recycled from `markers.py` / annotation metadata (no marker-score medoid).
`root_state`, `terminal_states`, `transient_states` come from the Hérault state hierarchy;
grouping uses `label` (`cell_type`).

In [ ]:
from markers import states_and_markers

ROOT_CANDIDATES = ("iHSC", "qHSC")  # initial / quiescent; prefer iHSC
TERMINAL_STATES = ("pEr", "pMk", "pLymph", "pNeuMast")
TRANSIENT_STATES = ("srHSC", "ifnHSC", "qHSC", "preDiff")

adata.obs["cell_type"] = adata.obs["label"].astype("category")
adata.obs["label"] = adata.obs["cell_type"]

present = set(adata.obs["label"].dropna().astype(str).unique())
root_state = next((s for s in ROOT_CANDIDATES if s in present), None)
if root_state is None:
    raise ValueError(f"No root candidate in labels {sorted(present)}; tried {ROOT_CANDIDATES}")

terminal_states = [s for s in TERMINAL_STATES if s in present]
transient_states = [s for s in TRANSIENT_STATES if s in present and s != root_state]

# Paper taxonomy (full Hérault set) — useful when annotation collapsed terminals
paper_present = set(adata.obs["paper_state"].dropna().astype(str).unique())
paper_root_state = next((s for s in ROOT_CANDIDATES if s in paper_present), None)
paper_terminals = [s for s in TERMINAL_STATES if s in paper_present]

adata.uns["root_state"] = root_state
adata.uns["terminal_states"] = terminal_states
adata.uns["transient_states"] = transient_states
adata.uns["ti_priors"] = {
    "layer": "A",
    "root_state": root_state,
    "terminal_states": terminal_states,
    "transient_states": transient_states,
    "root_candidates": list(ROOT_CANDIDATES),
    "states_and_markers_keys": list(states_and_markers),
    "paper_root_state": paper_root_state,
    "paper_terminal_states": paper_terminals,
}

print("root_state:", root_state, f"(n={int((adata.obs['label'] == root_state).sum())})")
print("terminal_states:", terminal_states)
print("transient_states:", transient_states)
print("paper_root_state:", paper_root_state, "| paper_terminals:", paper_terminals)
adata.uns["ti_priors"]

### Layer B — root cell for APIs that need it (DPT `iroot`, etc.)

1. Prefer medoid of `root_state` cells in integrated embedding (`X_emb` → `X_pca`, else existing `X_pca`).
2. If no embedding yet: use paper `hscScore` (argmax) / `paper_pseudotime` (argmin) within `root_state`.
3. Store `uns['root_cell']` and integer `uns['iroot']` for scanpy.

In [ ]:
from scipy.spatial.distance import cdist

root_mask = adata.obs["label"].astype(str) == adata.uns["root_state"]
root_idx = np.where(root_mask.to_numpy())[0]
if root_idx.size == 0:
    raise ValueError(f"No cells with label == {adata.uns['root_state']!r}")

# Embedding for medoid (plan: X_emb copied to X_pca for anndata2ri / Slingshot)
if "X_emb" in adata.obsm:
    adata.obsm["X_pca"] = np.asarray(adata.obsm["X_emb"])
emb_key = "X_pca" if "X_pca" in adata.obsm else None

root_cell = None
root_method = None

if emb_key is not None:
    coords = np.asarray(adata.obsm[emb_key])[root_idx]
    centroid = coords.mean(axis=0, keepdims=True)
    medoid_local = int(cdist(coords, centroid).argmin())
    root_cell = adata.obs_names[root_idx[medoid_local]]
    root_method = f"medoid:{emb_key}"
else:
    sub = adata.obs.loc[root_mask, ["hscScore", "paper_pseudotime"]].copy()
    if sub["hscScore"].notna().any():
        root_cell = sub["hscScore"].astype(float).idxmax()
        root_method = "argmax:hscScore"
    elif sub["paper_pseudotime"].notna().any():
        root_cell = sub["paper_pseudotime"].astype(float).idxmin()
        root_method = "argmin:paper_pseudotime"
    else:
        root_cell = adata.obs_names[root_idx[0]]
        root_method = "fallback:first_root_cell"
        print("WARNING: no embedding and no paper scores; using first root cell")

iroot = int(adata.obs_names.get_loc(root_cell))
adata.uns["root_cell"] = root_cell
adata.uns["iroot"] = iroot
adata.uns["ti_priors_B"] = {
    "layer": "B",
    "root_cell": root_cell,
    "iroot": iroot,
    "method": root_method,
    "n_root_cells": int(root_idx.size),
    "embedding_key": emb_key,
}

# Convenience aliases for later adapters
start_cluster = adata.uns["root_state"]
start_id = root_cell
end_clusters = list(adata.uns["terminal_states"])

print(f"root_cell={root_cell!r}  iroot={iroot}  via {root_method}")
print(f"start_cluster={start_cluster!r}  end_clusters={end_clusters}")
if root_cell in adata.obs.index:
    print(adata.obs.loc[root_cell, ["label", "paper_state", "hscScore", "paper_pseudotime"]])
adata.uns["ti_priors_B"]

## Inference

### PAGA

In [ ]:
# PAGA + DPT (jak tests/test_ti_methods_smoke.py), z iroot z Layer B
if "X_emb" in adata.obsm and "X_pca" not in adata.obsm:
    adata.obsm["X_pca"] = np.asarray(adata.obsm["X_emb"])
if "X_pca" not in adata.obsm:
    sc.pp.pca(adata, n_comps=50)

sc.pp.neighbors(adata, n_neighbors=15, use_rep="X_pca")
sc.tl.paga(adata, groups="label")
# adata.uns["iroot"] already set in Layer B
sc.tl.diffmap(adata)
sc.tl.dpt(adata)

adata.obs["pseudotime"] = adata.obs["dpt_pseudotime"]
adata.obs["branch"] = adata.obs["label"].astype("category")
print("dpt_pseudotime:", adata.obs["dpt_pseudotime"].describe()[["min", "50%", "max"]].to_dict())
adata.obs[["label", "dpt_pseudotime"]].groupby("label", observed=True)["dpt_pseudotime"].median().sort_values()

In [ ]:
sc.pl.paga(adata, color="label", title="PAGA (label)")
if "X_umap" not in adata.obsm:
    sc.tl.umap(adata)
sc.pl.umap(adata, color=["label", "dpt_pseudotime"], wspace=0.35)